# Deteksi Anomali Gempa Bumi BMKG
## Isolation Forest + Interpretasi SHAP

**Tujuan:** Mendeteksi gempa bumi yang karakteristiknya (magnitudo, kedalaman, lokasi) menyimpang dari pola historis gempa di Indonesia, sekaligus **menjelaskan** fitur mana yang menjadi penyebab utama sebuah gempa ditandai anomali.

**Data:** Katalog gempa BMKG Indonesia periode 2008–2025 (`gabungan_2008_2025.csv`).

**Metode:**
1. **Isolation Forest** (Liu, Ting & Zhou, 2008) — algoritma deteksi anomali *unsupervised* yang bekerja dengan prinsip: titik data anomali lebih mudah "diisolasi" (dipisahkan dari data lain) sehingga membutuhkan lebih sedikit partisi acak. Skor `decision_function` bertanda: **negatif = anomali, positif = normal**.
2. **SHAP TreeExplainer** (Lundberg & Lee, 2017; Lundberg et al., 2020) — metode *explainable AI* yang menghitung kontribusi eksak tiap fitur terhadap skor anomali, langsung dari struktur pohon model (bukan aturan heuristik terpisah). Fitur dengan nilai SHAP paling negatif adalah pendorong terbesar sebuah gempa ke arah anomali.

**Keluaran (disimpan ke folder `models/`):**
| File | Isi |
|---|---|
| `isolation_forest_bmkg.pkl` | Model Isolation Forest terlatih |
| `scaler_isolation_forest_bmkg.pkl` | StandardScaler (menyimpan mean & std data training — dipakai juga sebagai pembanding "rata-rata historis" di aplikasi) |
| `IF_SHAP.pkl` | DataFrame gempa anomali beserta nilai SHAP, fitur dominan, dan ranking kontribusi |
| `IF_SHAP_explainer.pkl` | Objek TreeExplainer (opsional, untuk menjelaskan gempa baru secara real-time di backend) |

## 1. Import Library dan Konfigurasi

`random_state=42` dikunci agar hasil training **reproducible** — setiap kali notebook dijalankan ulang dengan data yang sama, model dan hasil deteksinya identik.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Konfigurasi eksperimen
RANDOM_STATE = 42
CONTAMINATION = 0.005  # proporsi data yang diasumsikan anomali (0.5%)
FEATURES = ["mag", "depth", "latitude", "longitude"]

# Lokasi data dan folder keluaran (relatif terhadap folder notebook ini)
DATA_PATH = Path("../data/bmkg/gabungan_2008_2025.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

print("shap:", shap.__version__)

## 2. Memuat Data

Katalog gempa BMKG 2008–2025 dengan 6 kolom: waktu kejadian, koordinat episentrum, kedalaman (km), magnitudo, dan nama wilayah.

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Jumlah baris : {len(df):,}")
print(f"Rentang waktu: {df['time'].min()} s/d {df['time'].max()}")
df.head()

## 3. Validasi dan Pembersihan Data

Dua langkah penjagaan kualitas data:
1. **Filter bounding box Indonesia** (lintang -11.0 s/d 6.0, bujur 95.0 s/d 141.0) — membuang entri katalog yang koordinatnya keliru/di luar wilayah studi.
2. **Hapus baris dengan nilai kosong** pada fitur yang dipakai model.

Indeks asli tiap baris disimpan di kolom `index` agar hasil deteksi tetap bisa dirunut kembali ke katalog sumber.

In [ ]:
# Batas koordinat geografis Indonesia (secara kasar / bounding box)
lat_min, lat_max = -11.0, 6.0
lon_min, lon_max = 95.0, 141.0

mask_bbox = (
    df["latitude"].between(lat_min, lat_max)
    & df["longitude"].between(lon_min, lon_max)
)
print(f"Baris di luar bounding box : {(~mask_bbox).sum()}")
print(f"Baris dengan nilai kosong  : {df[FEATURES].isna().any(axis=1).sum()}")

df = df[mask_bbox].dropna(subset=FEATURES).reset_index()
print(f"Jumlah baris setelah pembersihan: {len(df):,}")

## 4. Eksplorasi Singkat

Statistik deskriptif keempat fitur. Nilai *mean* dan *std* di sini nantinya tersimpan di dalam StandardScaler dan menjadi acuan "pola historis" saat menjelaskan anomali di aplikasi.

In [ ]:
df[FEATURES].describe().round(4)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, kolom in zip(axes, FEATURES):
    ax.hist(df[kolom], bins=60, color="#4C72B0", edgecolor="white", linewidth=0.3)
    ax.set_title(kolom)
    ax.set_ylabel("frekuensi" if kolom == FEATURES[0] else "")
fig.suptitle("Distribusi Fitur Katalog Gempa BMKG 2008\u20132025", y=1.05)
plt.tight_layout()
plt.show()

## 5. Standardisasi Fitur

Keempat fitur memiliki skala yang sangat berbeda (magnitudo ~1–8, kedalaman ~1–650 km, koordinat derajat). **StandardScaler** mengubah tiap fitur menjadi z-score (rata-rata 0, simpangan baku 1) agar tidak ada fitur yang mendominasi hanya karena skalanya besar.

Scaler di-*fit* pada DataFrame (bukan array) agar nama fitur ikut tersimpan di `feature_names_in_`.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])

pd.DataFrame(
    {"mean (rata-rata historis)": scaler.mean_, "std (simpangan baku)": scaler.scale_},
    index=FEATURES,
).round(4)

## 6. Training Isolation Forest

Hyperparameter:
- `n_estimators=100` — jumlah pohon isolasi (default yang direkomendasikan paper aslinya sudah konvergen di ~100 pohon).
- `contamination=0.005` — asumsi 0.5% data adalah anomali; menentukan ambang batas skor untuk label anomali/normal.
- `random_state=42` — reproducibility.

In [ ]:
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
)
iso_forest.fit(X)

## 7. Hasil Deteksi

- `decision_function` → skor anomali bertanda (**negatif = anomali**, makin negatif makin ekstrem).
- `predict` → label: -1 (anomali) / 1 (normal). Label ini ekuivalen dengan memeriksa tanda skor terhadap ambang `contamination`.

In [ ]:
df["anomaly_score"] = iso_forest.decision_function(X)
df["is_anomaly"] = iso_forest.predict(X) == -1

n_anomali = int(df["is_anomaly"].sum())
print(f"Gempa terdeteksi anomali: {n_anomali:,} dari {len(df):,} "
      f"({n_anomali / len(df):.2%})")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(df.loc[~df["is_anomaly"], "anomaly_score"], bins=80,
        color="#4C72B0", label="Normal")
ax.hist(df.loc[df["is_anomaly"], "anomaly_score"], bins=80,
        color="#C44E52", label="Anomali")
ax.axvline(0, color="black", linestyle="--", linewidth=1,
           label="ambang keputusan (skor = 0)")
ax.set_xlabel("anomaly score (decision_function)")
ax.set_ylabel("frekuensi (log)")
ax.set_yscale("log")
ax.set_title("Distribusi Skor Anomali Isolation Forest")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Interpretasi dengan SHAP

`shap.TreeExplainer` menghitung kontribusi tiap fitur terhadap skor anomali **per gempa** (penjelasan lokal), langsung dari struktur pohon Isolation Forest. Konvensi tanda mengikuti skornya:

- Nilai SHAP **negatif** → fitur tersebut mendorong gempa ke arah **anomali**.
- Nilai SHAP **positif** → fitur tersebut mendorong ke arah **normal**.

Untuk tiap gempa anomali disusun: `shap_values` (dict), `ranking` (fitur terurut dari kontribusi paling negatif), dan `dominant_feature` (peringkat pertama).

In [ ]:
explainer = shap.TreeExplainer(iso_forest)

# Hitung SHAP hanya untuk gempa anomali (hemat komputasi)
mask_anomali = df["is_anomaly"].to_numpy()
X_anomali = X[mask_anomali]
shap_matrix = explainer.shap_values(X_anomali)

anomali_df = df[df["is_anomaly"]].copy()
anomali_df["shap_values"] = [dict(zip(FEATURES, baris)) for baris in shap_matrix]
anomali_df["ranking"] = [
    sorted(d.items(), key=lambda item: item[1])  # paling negatif lebih dulu
    for d in anomali_df["shap_values"]
]
anomali_df["dominant_feature"] = [r[0][0] for r in anomali_df["ranking"]]

print("Fitur dominan penyebab anomali:")
print(anomali_df["dominant_feature"].value_counts().to_string())

### 8.1 Uji Konsistensi Nilai SHAP

Properti *local accuracy* SHAP menyatakan jumlah nilai SHAP per instance merekonstruksi skor model. Catatan implementasi: untuk `IsolationForest`, `TreeExplainer` menghitung nilai SHAP pada skala **skor internal model** (rata-rata panjang lintasan isolasi antar pohon), yang merupakan transformasi monoton-linier dari `decision_function`. Karena itu validasi dilakukan lewat **korelasi** antara jumlah SHAP dan `anomaly_score` — nilai ≈ 1.0 membuktikan nilai SHAP konsisten dengan skor model (urutan dan proporsi kontribusi fiturnya sahih), meski satuannya berbeda.

In [ ]:
sum_shap = shap_matrix.sum(axis=1)
korelasi = np.corrcoef(sum_shap, anomali_df["anomaly_score"])[0, 1]
print(f"Korelasi sum(SHAP) vs anomaly_score: {korelasi:.6f}")
assert korelasi > 0.999, "nilai SHAP tidak konsisten dengan skor model!"

In [ ]:
# Rata-rata |SHAP| per fitur: gambaran global fitur mana yang paling berperan
shap.summary_plot(shap_matrix, features=X_anomali, feature_names=FEATURES,
                  plot_type="bar", show=False)
plt.title("Kontribusi Global Fitur terhadap Anomali (rata-rata |SHAP|)")
plt.tight_layout()
plt.show()

### 8.2 Contoh Pembacaan Satu Gempa Anomali

Contoh cara membaca hasil untuk gempa paling anomali (skor terendah) — pola narasi yang sama dipakai aplikasi saat menjelaskan anomali ke pengguna.

In [ ]:
contoh = anomali_df.sort_values("anomaly_score").iloc[0]

print(f"Waktu     : {contoh['time']}")
print(f"Wilayah   : {contoh['wilayah']}")
print(f"Magnitudo : {contoh['mag']:.2f}  (historis: {scaler.mean_[0]:.2f} \u00b1 {scaler.scale_[0]:.2f})")
print(f"Kedalaman : {contoh['depth']:.0f} km  (historis: {scaler.mean_[1]:.1f} \u00b1 {scaler.scale_[1]:.1f} km)")
print(f"Skor      : {contoh['anomaly_score']:.4f}")
print(f"Faktor dominan: {contoh['dominant_feature']}")
print("Ranking kontribusi (paling negatif = paling mendorong anomali):")
for fitur, nilai in contoh["ranking"]:
    print(f"  {fitur:<10} {nilai:+.4f}")

## 9. Menyimpan Artefak

Empat artefak disimpan ke `models/` dengan `joblib`. Susunan kolom `IF_SHAP.pkl` mengikuti format yang dikonsumsi backend aplikasi.

In [ ]:
KOLOM_IF_SHAP = [
    "index", "time", "mag", "depth", "latitude", "longitude", "wilayah",
    "anomaly_score", "is_anomaly", "shap_values", "dominant_feature", "ranking",
]
if_shap_df = anomali_df[KOLOM_IF_SHAP].reset_index(drop=True)

joblib.dump(iso_forest, MODEL_DIR / "isolation_forest_bmkg.pkl")
joblib.dump(scaler, MODEL_DIR / "scaler_isolation_forest_bmkg.pkl")
joblib.dump(if_shap_df, MODEL_DIR / "IF_SHAP.pkl")
joblib.dump(explainer, MODEL_DIR / "IF_SHAP_explainer.pkl")

for f in sorted(MODEL_DIR.glob("*.pkl")):
    print(f"{f.name:<40} {f.stat().st_size / 1024:,.0f} KB")

## 10. Verifikasi Artefak

Muat ulang semua artefak dari disk dan pastikan hasilnya konsisten dengan sesi training — memastikan file yang dibawa ke backend memang benar.

In [ ]:
model_v = joblib.load(MODEL_DIR / "isolation_forest_bmkg.pkl")
scaler_v = joblib.load(MODEL_DIR / "scaler_isolation_forest_bmkg.pkl")
if_shap_v = joblib.load(MODEL_DIR / "IF_SHAP.pkl")

X_v = scaler_v.transform(df[FEATURES])
assert (model_v.predict(X_v) == -1).sum() == n_anomali, "jumlah anomali berubah!"
assert len(if_shap_v) == n_anomali, "jumlah baris IF_SHAP tidak cocok!"
assert np.allclose(model_v.decision_function(X_v), df["anomaly_score"]), "skor berubah!"

print(f"Verifikasi OK \u2014 {n_anomali:,} anomali, skor identik, artefak siap dipakai backend.")

## Referensi

1. Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). *Isolation Forest*. IEEE International Conference on Data Mining (ICDM).
2. Lundberg, S. M., & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions*. Advances in Neural Information Processing Systems (NeurIPS).
3. Lundberg, S. M., et al. (2020). *From local explanations to global understanding with explainable AI for trees*. Nature Machine Intelligence, 2(1), 56–67.
4. Badan Meteorologi, Klimatologi, dan Geofisika (BMKG). Katalog gempa bumi Indonesia. https://data.bmkg.go.id